# Fraud Detection - Self-Contained Project Notebook

## Objective
Train and evaluate an imbalanced-class fraud model with local dataset support and synthetic fallback data.

## Expected input schema
`Class` target plus numeric feature columns (for example `Time`, `Amount`, `V1...Vn`).

## Output artifacts
- `output/fraud_metrics.csv`
- `output/validation_scored.csv`

In [1]:
import numpy as np
import pandas as pd
from pathlib import Path

SEED = 42
rng = np.random.default_rng(SEED)

project_dir = Path.cwd()
if not (project_dir / "Fraud_Detection.ipynb").exists():
    project_dir = Path.cwd() / "HandsOn-Projects" / "Fraud_Detection_Project"

data_dir = project_dir / "data"
output_dir = project_dir / "output"
data_dir.mkdir(parents=True, exist_ok=True)
output_dir.mkdir(parents=True, exist_ok=True)

DATA_PATH = data_dir / "creditcard.csv"

def make_synthetic_fraud(n_rows: int = 60000, fraud_rate: float = 0.012) -> pd.DataFrame:
    n_fraud = int(n_rows * fraud_rate)
    n_legit = n_rows - n_fraud

    legit = pd.DataFrame({
        "Time": rng.integers(0, 172800, size=n_legit),
        "Amount": np.clip(rng.gamma(shape=2.0, scale=35.0, size=n_legit), 0, 3000),
    })
    fraud = pd.DataFrame({
        "Time": rng.integers(0, 172800, size=n_fraud),
        "Amount": np.clip(rng.gamma(shape=2.8, scale=120.0, size=n_fraud), 0, 6000),
    })

    for i in range(1, 11):
        legit[f"V{i}"] = rng.normal(0, 1, size=n_legit)
        fraud[f"V{i}"] = rng.normal(0.8 if i % 2 == 0 else -0.8, 1.4, size=n_fraud)

    legit["Class"] = 0
    fraud["Class"] = 1
    out = pd.concat([legit, fraud], ignore_index=True).sample(frac=1, random_state=SEED).reset_index(drop=True)
    return out

if DATA_PATH.exists():
    df = pd.read_csv(DATA_PATH)
else:
    df = make_synthetic_fraud()
    df.to_csv(DATA_PATH, index=False)

print("Project dir:", project_dir)
print("Data path:", DATA_PATH)
print("Rows:", len(df))
df.head()

Project dir: c:\Users\Moshe\Documents\GitHub\Data-Engineering-Prep-Guide\HandsOn-Projects\Fraud_Detection_Project
Data path: c:\Users\Moshe\Documents\GitHub\Data-Engineering-Prep-Guide\HandsOn-Projects\Fraud_Detection_Project\data\creditcard.csv
Rows: 60000


,Time,Amount,V1,V2,V3,V4,V5,V6,V7,V8,V9,V10,Class
0,24007,52.132073,0.323142,0.100083,-0.229778,-0.919376,1.338631,-0.891347,-1.223512,0.137430,0.685094,0.529318,0
1,37714,54.036763,0.200820,-1.019743,0.058832,0.624356,0.869809,-1.024099,-1.804370,-1.101113,-1.027233,-0.423228,0
2,140373,50.116297,-1.385058,-0.114919,0.175319,0.514252,0.775293,0.631904,-0.516687,-0.861120,-1.361836,0.778214,0
3,48925,24.935440,-1.384895,1.771922,-0.077384,-0.899858,-1.241083,-0.362053,-0.550581,-0.236000,-2.061720,-0.304921,0
4,137498,19.030926,0.372932,1.166232,0.463486,0.436785,-0.185713,0.305849,-0.333808,0.985552,-0.069353,0.460587,0


## 1) EDA and split strategy

### Why this matters
Fraud detection is highly imbalanced, so stratified splitting and class-aware metrics are mandatory.

This section:
- Validates target column
- Checks class imbalance
- Creates stratified train/validation splits

In [2]:
from sklearn.model_selection import train_test_split

target = "Class"
if target not in df.columns:
    raise ValueError("Missing target column 'Class'.")

feature_cols = [c for c in df.columns if c != target]
X = df[feature_cols].copy()
y = df[target].astype(int).copy()

print("Class distribution:")
print(y.value_counts(normalize=True).rename("ratio"))

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=SEED
)

print("Train shape:", X_train.shape, "Valid shape:", X_valid.shape)

ModuleNotFoundError: No module named 'sklearn'

## 2) Baseline model and threshold tuning

We train a class-weighted logistic model, evaluate ROC/PR AUC, and tune thresholds using expected business cost.

In [3]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve

model = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=SEED)),
])
model.fit(X_train, y_train)

proba = model.predict_proba(X_valid)[:, 1]
roc_auc = roc_auc_score(y_valid, proba)
pr_auc = average_precision_score(y_valid, proba)
print("ROC AUC:", round(roc_auc, 4))
print("PR AUC:", round(pr_auc, 4))

# Cost-based threshold tuning:
# false positive investigation cost = $10
# missed fraud cost = $500
precision, recall, thresholds = precision_recall_curve(y_valid, proba)
thr = np.append(thresholds, 1.0)

records = []
yv = y_valid.to_numpy()
for t in thr:
    pred = (proba >= t).astype(int)
    fp = int(((pred == 1) & (yv == 0)).sum())
    fn = int(((pred == 0) & (yv == 1)).sum())
    tp = int(((pred == 1) & (yv == 1)).sum())
    tn = int(((pred == 0) & (yv == 0)).sum())
    total_cost = fp * 10 + fn * 500
    records.append((t, fp, fn, tp, tn, total_cost))

cost_df = pd.DataFrame(records, columns=["threshold", "FP", "FN", "TP", "TN", "total_cost"]).sort_values("total_cost")
best_row = cost_df.iloc[0]
best_t = float(best_row["threshold"])
print("Best threshold:", round(best_t, 4), "with total cost:", int(best_row["total_cost"]))

ModuleNotFoundError: No module named 'sklearn'

## 3) Persist scored outputs and monitoring baseline

This section exports validation predictions and key metrics so this folder can function as a standalone repository artifact.

In [4]:
scored_valid = X_valid.copy()
scored_valid["actual_class"] = y_valid.values
scored_valid["fraud_proba"] = proba
scored_valid["pred_class_optimal"] = (proba >= best_t).astype(int)

metrics_df = pd.DataFrame([
    {"metric": "roc_auc", "value": float(roc_auc)},
    {"metric": "pr_auc", "value": float(pr_auc)},
    {"metric": "best_threshold", "value": float(best_t)},
    {"metric": "min_total_cost", "value": float(best_row["total_cost"])}
])

scored_valid.to_csv(output_dir / "validation_scored.csv", index=False)
metrics_df.to_csv(output_dir / "fraud_metrics.csv", index=False)

print("Saved:", output_dir / "validation_scored.csv")
print("Saved:", output_dir / "fraud_metrics.csv")
metrics_df

NameError: name 'X_valid' is not defined